<a href="https://colab.research.google.com/github/HLZHarry/Alpha-Lens-TPM/blob/main/03_AI_Macro_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this phase, we are solving a classic institutional problem: Market data is "Lagged" (Backward-looking), but Central Bank text is "Forward-looking."

We will build an agent that reads a simulated "Federal Reserve Statement" and converts the "Hawkish" (High Interest Rate) or "Dovish" (Low Interest Rate) tone into a numerical Regime Score.

Enhanced with RAG solution:
1. The Problem: FOMC Statements are short. The real "Alpha" is often hidden in the Minutes (released 3 weeks later) or in how the tone has changed compared to the previous meeting.
2. The RAG Solution: We will store all historical Statements and Minutes in a Vector Database (ChromaDB). When we score a date, the system "retrieves" the most recent documents to give the AI context.

In [ ]:
!pip install -U openai chromadb

In [ ]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
import numpy as np
import time
import os
from google.colab import userdata, drive

In [ ]:
# 1. INITIALIZE & MOUNT DRIVE
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Alpha-Lens-Project"
os.makedirs(f"{path}/chroma_v3", exist_ok=True)

In [ ]:
# 2. SETUP THE STACK
# Using FinBERT for specialized financial understanding
finbert_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="ProsusAI/finbert")
client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

# PERSISTENT ChromaDB: Saves the database to your Drive
chroma_client = chromadb.PersistentClient(path=f"{path}/chroma_v3")
collection = chroma_client.get_or_create_collection(
    name="fomc_intelligence",
    embedding_function=finbert_ef
)

In [ ]:
# 3. COMPLETE MEETING DATES (2021-2026)
meeting_dates = [
    '20210127', '20210317', '20210428', '20210616', '20210728', '20210922', '20211103', '20211215',
    '20220126', '20220316', '20220504', '20220615', '20220727', '20220921', '20221102', '20221214',
    '20230201', '20230322', '20230503', '20230614', '20230726', '20230920', '20231101', '20231213',
    '20240131', '20240320', '20240501', '20240612', '20240731', '20240918', '20241107', '20241218',
    '20250129', '20250319', '20250507', '20250618', '20250730', '20250917', '20251029', '20251210',
    '20260128'
]

In [ ]:
# 4. RESILIENT SCRAPER
def scrape_fed_docs(date):
    headers = {'User-Agent': 'Mozilla/5.0'}
    combined_docs = []
    # Scraping Statement + Minutes
    urls = {
        "S": f"https://www.federalreserve.gov/newsevents/pressreleases/monetary{date}a.htm",
        "M": f"https://www.federalreserve.gov/monetarypolicy/fomcminutes{date}.htm"
    }
    for key, url in urls.items():
        try:
            r = requests.get(url, headers=headers, timeout=10)
            if r.status_code == 200:
                soup = BeautifulSoup(r.content, 'html.parser')
                text = " ".join([p.text for p in soup.find_all('p') if len(p.text) > 100])
                if len(text) > 200:
                    combined_docs.append(text)
        except: continue
    return "\n---\n".join(combined_docs)

In [ ]:
# 5. EXECUTION & SCORING LOOP
print("🚀 Starting Production RAG Pipeline...")
results = []

for date in meeting_dates:
    print(f"Processing {date}...", end=" ")

    # A. Scrape Text
    full_text = scrape_fed_docs(date)
    if not full_text or len(full_text) < 100:
        print("❌ Scrape Failed (Empty)")
        results.append({'Date': pd.to_datetime(date), 'AI_Sentiment': 0.0})
        continue

    # B. Store in Vector DB (Upsert)
    collection.upsert(
        documents=[full_text[:4000]],
        metadatas=[{"date": date}],
        ids=[f"id_{date}"]
    )

    # C. RAG STEP: Retrieve context from the Database
    # This retrieves the most similar previous meetings to compare tone
    history = collection.query(query_texts=["inflation, interest rates, and employment outlook"], n_results=2)
    context_text = " ".join(history['documents'][0])

    # D. ANALYZE: GPT-4o with Context
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a Senior TPM Analyst. Compare the current text to historical context for policy shifts."},
                {"role": "user", "content": f"HISTORICAL CONTEXT:\n{context_text[:1500]}\n\nCURRENT STATEMENT:\n{full_text[:2000]}\n\nTASK: Score the sentiment from -1.0 (Hawkish) to 1.0 (Dovish). Return ONLY the float."}
            ],
            temperature=0
        )
        score = float(response.choices[0].message.content.strip())
        results.append({'Date': pd.to_datetime(date), 'AI_Sentiment': score})
        print(f"✅ Score: {score}")
    except Exception as e:
        print(f"⚠️ Error: {e}")
        results.append({'Date': pd.to_datetime(date), 'AI_Sentiment': 0.0})

    time.sleep(1) # Throttling

In [ ]:
# 6. INTEGRATE & SAVE
sentiment_df = pd.DataFrame(results)
df_features = pd.read_csv(f"{path}/engineered_features_v2.csv")
df_features['Date'] = pd.to_datetime(df_features['Date'])

df_final = pd.merge(df_features, sentiment_df, on='Date', how='left')
# Propagation: The Fed's mood lasts until the next meeting
df_final['AI_Sentiment'] = df_final.groupby('Ticker')['AI_Sentiment'].ffill().fillna(0)

df_final.to_csv(f"{path}/final_hybrid_data_v3.csv", index=False)
print("\n✅ Phase 3 Complete. Final dataset saved as 'final_hybrid_data_v3.csv'")

# 🏛️ Phase 3 Code Breakdown: Steps #4 and #5

---

## Step #4: The Resilient Scraper & RAG Ingestion

This section is the **"Data Ingestion" layer**. Its job is to find the text on the web and store it in our "Vector Memory" (ChromaDB).

### `scrape_fed_docs(date)` function

- **`headers = {'User-Agent': 'Mozilla/5.0'}`** — Mimics a real web browser. Without this, the Federal Reserve server might identify the script as a "bot" and block the connection.

- **`urls = {"S": ..., "M": ...}`** — Defines two sources for every date: the Statement (`S`) and the Minutes (`M`).

- **`requests.get(url, ...)`** — Sends the actual request to the website to download the content.

- **`BeautifulSoup(r.content, 'html.parser')`** — "Parses" the raw HTML code, allowing us to pick out specific tags.

- **`[p.text for p in soup.find_all('p') if len(p.text) > 100]`** — A list comprehension that grabs all paragraphs (`<p>` tags) longer than 100 characters, filtering out "noise" like menu buttons or footer links.

---

## Step #5: The Execution & RAG Scoring Loop

This is the **"Reasoning" layer**. It connects the data we just scraped with the AI's ability to analyze it in context.

### `collection.upsert(...)`

Instead of just "adding" data, **Upsert (Update + Insert)** checks if an ID (like `id_20260128`) already exists. If it does, it updates it; if not, it creates it. This prevents duplicate data in your database if you run the cell multiple times.


### `history = collection.query(query_texts=[...], n_results=2)`

This is the **Retrieval** part of RAG. We ask the database: *"Find the 2 documents in our history that are most mathematically similar to this topic."* It provides the "context" for the AI.

### `client.chat.completions.create(...)`

- **`model="gpt-4o"`** — Specifies the "Brain" we are using.

- **`messages`**:
  - **`role: "system"`** — Sets the "persona" of the AI (e.g., *"You are a Senior Analyst"*).
  - **`role: "user"`** — Contains the actual prompt, where we inject the **Context** (past data) and the **Current** (today's data).

- **`temperature=0`** — Makes the AI "deterministic." Ensures that if you run the same text through twice, you get the same score — critical for financial consistency.

### `df_final.groupby('Ticker')['AI_Sentiment'].ffill()`

**FFILL (Forward Fill):** Fed meetings only happen 8 times a year. This command takes the score from a meeting date and "stretches" it forward to every following day until the next meeting date occurs. This ensures our model always has a "Macro Regime" value for every trading day.